In [1]:



# ============================================================================
# STEP 1: Install Dependencies
# ============================================================================
print("📦 Installing packages...")
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "gradio", "groq"])
print("✅ Packages installed!\n")

# ============================================================================
# STEP 2: Import Libraries
# ============================================================================
import pandas as pd
import gradio as gr
from datetime import datetime
import json
from groq import Groq
from typing import Dict, List, Any

# ============================================================================
# STEP 3: Agent Base Class
# ============================================================================

class Agent:
    """Base Agent class with LLM capabilities"""

    def __init__(self, name: str, role: str, groq_client: Groq):
        self.name = name
        self.role = role
        self.groq_client = groq_client
        self.logs = []

    def log(self, message: str):
        """Log agent activity"""
        log_entry = f"[{self.name}] {message}"
        self.logs.append(log_entry)
        print(log_entry)

    def call_llm(self, system_prompt: str, user_prompt: str) -> str:
        """Call Groq LLM"""
        try:
            response = self.groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.3,
                max_tokens=500
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            self.log(f"LLM Error: {str(e)}")
            return "{}"

# ============================================================================
# STEP 4: Specialized Agent Classes
# ============================================================================

class ClassifierAgent(Agent):
    """Classifies feedback using LLM"""

    def __init__(self, groq_client: Groq):
        super().__init__("Classifier Agent", "Feedback Classification", groq_client)

    def classify(self, feedback_text: str) -> Dict:
        """Classify feedback into categories"""
        self.log(f"Classifying: {feedback_text[:60]}...")

        system = """You are a feedback classification expert. Classify into ONE category:
- Bug (crashes, errors, broken features)
- Feature Request (new features, improvements)
- Praise (positive feedback)
- Complaint (negative feedback)
- Spam (promotional/gibberish)

Return ONLY valid JSON:
{"category": "Bug", "confidence": 0.9, "reason": "describes crash"}"""

        response = self.call_llm(system, f"Classify:\n{feedback_text}")

        try:
            result = json.loads(response)
            self.log(f"✓ Category: {result.get('category', 'Unknown')}")
            return result
        except:
            return {"category": "General", "confidence": 0.5, "reason": "Parse error"}


class BugAnalysisAgent(Agent):
    """Analyzes bugs and assigns priority"""

    def __init__(self, groq_client: Groq):
        super().__init__("Bug Analysis Agent", "Bug Priority Analysis", groq_client)

    def analyze(self, feedback_text: str) -> Dict:
        """Analyze bug and determine priority"""
        self.log(f"Analyzing bug...")

        system = """Analyze bug severity. Return ONLY valid JSON:
{"priority": "Critical/High/Medium/Low", "severity": "description", "impact": "user impact"}

Priority rules:
- Critical: data loss, auth failures, crashes
- High: major functionality broken
- Medium: minor issues
- Low: cosmetic issues"""

        response = self.call_llm(system, f"Bug:\n{feedback_text}")

        try:
            result = json.loads(response)
            self.log(f"✓ Priority: {result.get('priority', 'Medium')}")
            return result
        except:
            return {"priority": "Medium", "severity": "Unknown", "impact": "Unknown"}


class FeatureAnalysisAgent(Agent):
    """Analyzes feature requests"""

    def __init__(self, groq_client: Groq):
        super().__init__("Feature Analysis Agent", "Feature Evaluation", groq_client)

    def analyze(self, feedback_text: str) -> Dict:
        """Analyze feature request"""
        self.log(f"Analyzing feature request...")

        system = """Analyze feature request. Return ONLY valid JSON:
{"feature": "name", "benefit": "user benefit", "priority": "High/Medium/Low"}"""

        response = self.call_llm(system, f"Feature request:\n{feedback_text}")

        try:
            result = json.loads(response)
            self.log(f"✓ Feature: {result.get('feature', 'Unknown')}")
            return result
        except:
            return {"feature": "Unknown", "benefit": "Unknown", "priority": "Medium"}


class TicketCreatorAgent(Agent):
    """Creates structured tickets"""

    def __init__(self, groq_client: Groq):
        super().__init__("Ticket Creator Agent", "Ticket Generation", groq_client)

    def create_ticket(self, feedback_data: Dict) -> Dict:
        """Create structured ticket"""
        self.log(f"Creating ticket...")

        system = """Create a clear ticket title. Format: [CATEGORY] Brief title
Examples: [BUG] App crashes on export, [FEATURE] Add dark mode
Return ONLY the title, max 60 chars."""

        prompt = f"Category: {feedback_data['category']}\nText: {feedback_data['text']}"
        title = self.call_llm(system, prompt).strip().strip('"')

        ticket = {
            'ticket_id': f"TKT-{datetime.now().strftime('%Y%m%d%H%M')}-{feedback_data['id']}",
            'title': title[:70],
            'category': feedback_data['category'],
            'priority': feedback_data.get('priority', 'Medium'),
            'description': feedback_data['text'][:200],
            'confidence': feedback_data.get('confidence', 0.8),
            'source': feedback_data['source'],
            'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }

        self.log(f"✓ Ticket created: {ticket['ticket_id']}")
        return ticket


class QualityReviewAgent(Agent):
    """Reviews ticket quality"""

    def __init__(self, groq_client: Groq):
        super().__init__("Quality Review Agent", "Quality Assurance", groq_client)

    def review(self, ticket: Dict) -> Dict:
        """Review ticket quality"""
        self.log(f"Reviewing ticket quality...")

        system = """Review ticket quality. Return ONLY valid JSON:
{"score": 8, "complete": true, "issues": []}
Score 1-10."""

        response = self.call_llm(system, f"Ticket: {json.dumps(ticket)}")

        try:
            review = json.loads(response)
            score = review.get('score', 7)
            self.log(f"✓ Quality score: {score}/10")
            ticket['quality_score'] = score
            return ticket
        except:
            ticket['quality_score'] = 7
            return ticket

# ============================================================================
# STEP 5: Multi-Agent Orchestrator
# ============================================================================

class MultiAgentOrchestrator:
    """Orchestrates all agents to process feedback"""

    def __init__(self, groq_api_key: str):
        self.groq_client = Groq(api_key=groq_api_key)

        # Initialize all agents
        self.classifier = ClassifierAgent(self.groq_client)
        self.bug_analyzer = BugAnalysisAgent(self.groq_client)
        self.feature_analyzer = FeatureAnalysisAgent(self.groq_client)
        self.ticket_creator = TicketCreatorAgent(self.groq_client)
        self.quality_reviewer = QualityReviewAgent(self.groq_client)

        self.all_tickets = []
        self.all_logs = []

    def process_feedback(self, id: str, text: str, source: str) -> Dict:
        """Process one feedback through agent pipeline"""

        print(f"\n{'='*70}")
        print(f"🤖 PROCESSING: {id}")
        print(f"{'='*70}")

        # Agent 1: Classify
        classification = self.classifier.classify(text)

        feedback_data = {
            'id': id,
            'text': text,
            'source': source,
            'category': classification.get('category', 'General'),
            'confidence': classification.get('confidence', 0.7)
        }

        # Agent 2 & 3: Specialized analysis
        if feedback_data['category'] == 'Bug':
            analysis = self.bug_analyzer.analyze(text)
            feedback_data['priority'] = analysis.get('priority', 'Medium')
        elif feedback_data['category'] == 'Feature Request':
            analysis = self.feature_analyzer.analyze(text)
            feedback_data['priority'] = analysis.get('priority', 'Medium')
        else:
            priority_map = {'Praise': 'Low', 'Complaint': 'Medium', 'Spam': 'Low'}
            feedback_data['priority'] = priority_map.get(feedback_data['category'], 'Low')

        # Agent 4: Create ticket
        ticket = self.ticket_creator.create_ticket(feedback_data)

        # Agent 5: Quality review
        final_ticket = self.quality_reviewer.review(ticket)

        self.all_tickets.append(final_ticket)

        # Collect logs
        for agent in [self.classifier, self.bug_analyzer, self.feature_analyzer,
                      self.ticket_creator, self.quality_reviewer]:
            self.all_logs.extend(agent.logs)
            agent.logs = []

        return final_ticket

# ============================================================================
# STEP 6: Sample Data
# ============================================================================

print("📄 Creating sample data...")

reviews = pd.DataFrame({
    'review_id': ['R001', 'R002', 'R003', 'R004', 'R005'],
    'text': [
        'App crashes when I export to PDF!',
        'Cannot login after update. Auth error.',
        'Please add dark mode feature!',
        'Love this app! Best productivity tool.',
        'App freezes on large file uploads.'
    ]
})

emails = pd.DataFrame({
    'email_id': ['E001', 'E002', 'E003', 'E004', 'E005'],
    'subject': ['Bug Report', 'Feature Request', 'Thank You', 'Performance Issue', 'Suggestion'],
    'body': [
        'App crashes on PDF export. iPhone 14 Pro, iOS 17.',
        'Would love offline mode for traveling.',
        'Thank you for the great update!',
        'App is very slow on Samsung Galaxy S21.',
        'Please integrate with Google Drive.'
    ]
})

reviews.to_csv('reviews.csv', index=False)
emails.to_csv('emails.csv', index=False)
print(f"✅ Created {len(reviews)} reviews + {len(emails)} emails\n")

# ============================================================================
# STEP 7: Gradio Interface
# ============================================================================

orchestrator = None

def setup_system(api_key: str):
    """Initialize system with Groq API key"""
    global orchestrator

    if not api_key or len(api_key.strip()) < 10:
        return "❌ Please enter a valid Groq API key!", None, None

    try:
        orchestrator = MultiAgentOrchestrator(api_key.strip())
        return "✅ System initialized! All 5 agents ready.", None, None
    except Exception as e:
        return f"❌ Error: {str(e)}\nCheck your API key!", None, None

def process_all_feedback():
    """Process all feedback through agents"""
    global orchestrator

    if not orchestrator:
        return "❌ Initialize system first!", None, "❌ Not ready"

    try:
        reviews_df = pd.read_csv('reviews.csv')
        emails_df = pd.read_csv('emails.csv')

        print("\n" + "="*80)
        print("🚀 STARTING MULTI-AGENT PROCESSING")
        print("="*80 + "\n")

        # Process reviews
        for _, row in reviews_df.iterrows():
            orchestrator.process_feedback(row['review_id'], row['text'], 'Review')

        # Process emails
        for _, row in emails_df.iterrows():
            text = f"{row['subject']}\n{row['body']}"
            orchestrator.process_feedback(row['email_id'], text, 'Email')

        # Create results DataFrame
        results = pd.DataFrame([{
            'Ticket ID': t['ticket_id'],
            'Category': t['category'],
            'Priority': t['priority'],
            'Title': t['title'],
            'Confidence': f"{t['confidence']:.0%}",
            'Quality': f"{t['quality_score']}/10",
            'Source': t['source']
        } for t in orchestrator.all_tickets])

        stats = {
            'Total': len(orchestrator.all_tickets),
            'Bugs': sum(1 for t in orchestrator.all_tickets if t['category'] == 'Bug'),
            'Features': sum(1 for t in orchestrator.all_tickets if t['category'] == 'Feature Request'),
        }

        summary = f"""
## ✅ Multi-Agent Processing Complete!

### 📊 Results
- **Total Tickets:** {stats['Total']}
- **🐛 Bugs:** {stats['Bugs']}
- **💡 Features:** {stats['Features']}

### 🤖 Agents Used
1. ✓ Classifier Agent
2. ✓ Bug Analysis Agent
3. ✓ Feature Analysis Agent
4. ✓ Ticket Creator Agent
5. ✓ Quality Review Agent

**Total Agent Actions:** {len(orchestrator.all_logs)}
"""

        pd.DataFrame(orchestrator.all_tickets).to_csv('tickets.csv', index=False)

        return summary, results, "✅ Done! Check tickets.csv"

    except Exception as e:
        return f"❌ Error: {str(e)}", None, "❌ Failed"

def show_logs():
    """Display agent logs"""
    if not orchestrator or not orchestrator.all_logs:
        return "No logs yet. Process feedback first!"

    recent = orchestrator.all_logs[-30:]
    return "### 🤖 Agent Activity Logs\n\n```\n" + "\n".join(recent) + "\n```"

# Create Interface
with gr.Blocks(title="Multi-Agent Feedback System", theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    # 🤖 Intelligent Feedback Analysis System
    ## Real Multi-Agent AI with Groq LLM

    **5 Specialized Agents Working Together:**
    1. Classifier Agent 🏷️
    2. Bug Analysis Agent 🐛
    3. Feature Analysis Agent 💡
    4. Ticket Creator Agent 🎫
    5. Quality Review Agent ✅
    """)

    with gr.Tab("🔑 Setup"):
        gr.Markdown("""
        ### Get Your FREE Groq API Key

        1. Go to: **https://console.groq.com**
        2. Sign up (free, takes 30 seconds)
        3. Click **"Create API Key"**
        4. Copy and paste below
        """)

        api_key = gr.Textbox(
            label="Your Groq API Key",
            placeholder="gsk_...",
            type="password",
            lines=1
        )

        setup_btn = gr.Button("🚀 Initialize System", variant="primary", size="lg")
        setup_status = gr.Textbox(label="Status")

        setup_btn.click(
            fn=setup_system,
            inputs=[api_key],
            outputs=[setup_status, gr.Textbox(visible=False), gr.Textbox(visible=False)]
        )

    with gr.Tab("📊 Process"):
        gr.Markdown("### Process Feedback with Multi-Agent System")

        process_btn = gr.Button("🤖 Start Processing", variant="primary", size="lg")
        process_status = gr.Textbox(label="Status")

        summary = gr.Markdown()
        tickets = gr.Dataframe(label="Generated Tickets")

        process_btn.click(
            fn=process_all_feedback,
            outputs=[summary, tickets, process_status]
        )

    with gr.Tab("📋 Agent Logs"):
        gr.Markdown("### Real-Time Agent Activity")

        refresh_btn = gr.Button("🔄 Refresh Logs")
        logs = gr.Markdown()

        refresh_btn.click(fn=show_logs, outputs=[logs])

    with gr.Tab("📄 Data"):
        gr.Markdown("### Sample Input Data")
        gr.Dataframe(value=reviews, label="Reviews")
        gr.Dataframe(value=emails, label="Emails")

print("\n" + "="*80)
print("✅ SYSTEM READY!")
print("="*80)

app.launch(share=True)

📦 Installing packages...
✅ Packages installed!

📄 Creating sample data...
✅ Created 5 reviews + 5 emails



C:\Users\HP INDIA\AppData\Local\Temp\ipykernel_13444\2741041882.py:393: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Multi-Agent Feedback System", theme=gr.themes.Soft()) as app:



✅ SYSTEM READY!
* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.



🚀 STARTING MULTI-AGENT PROCESSING


🤖 PROCESSING: R001
[Classifier Agent] Classifying: App crashes when I export to PDF!...
[Classifier Agent] ✓ Category: Bug
[Bug Analysis Agent] Analyzing bug...
[Bug Analysis Agent] ✓ Priority: Critical
[Ticket Creator Agent] Creating ticket...
[Ticket Creator Agent] ✓ Ticket created: TKT-202605111918-R001
[Quality Review Agent] Reviewing ticket quality...
[Quality Review Agent] ✓ Quality score: 9/10

🤖 PROCESSING: R002
[Classifier Agent] Classifying: Cannot login after update. Auth error....
[Classifier Agent] ✓ Category: Bug
[Bug Analysis Agent] Analyzing bug...
[Bug Analysis Agent] ✓ Priority: Critical
[Ticket Creator Agent] Creating ticket...
[Ticket Creator Agent] ✓ Ticket created: TKT-202605111918-R002
[Quality Review Agent] Reviewing ticket quality...
[Quality Review Agent] ✓ Quality score: 9/10

🤖 PROCESSING: R003
[Classifier Agent] Classifying: Please add dark mode feature!...
[Classifier Agent] ✓ Category: Feature Request
[Feature Analysis